# Implementação da Arquitetura LeNet-5 em PyTorch

In [5]:
import torch
import torch.nn as nn

class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        
        # ---  1. EXTRAÇÃO DE CARACTERÍSTICAS (FEATURES) ---
        # Esta parte da rede usa Convoluções para "enxergar" a imagem, procurando por 
        # padrões como bordas, cantos e texturas.
        # A rede original esperava imagens em escala de cinza e com tamanho 32x32 pixels.
        self.features = nn.Sequential(
            
            # 1ª CAMADA CONVOLUCIONAL:
            # in_channels=1: a imagem de entrada tem apenas 1 canal de cor (escala de cinza).
            # out_channels=6: estamos criando 6 "filtros" que vão produzir 6 representações (ou mapas) diferentes da imagem.
            # kernel_size=5: cada filtro olha para a imagem em pedaços (janelas) de 5x5 pixels.
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1),
            
            # FUNÇÃO DE ATIVAÇÃO (ReLU):
            # Redes neurais precisam de matemática não-linear para aprender padrões abstratos e complexos.
            # Sem funções de ativação, o modelo seria apenas uma sequência de contas lineares simples.
            # ReLU resolve o problema do gradiente de desvanecimento e é mais rápida de calcular.
            nn.ReLU(),
            
            # 1º POOLING (Subamostragem via Max Pooling):
            # Ela diminui o tamanho espacial (largura e altura) da imagem pela metade. Isso torna a rede mais rápida 
            # computacionalmente e também ajuda a IA a reconhecer o padrão mesmo que ele mude um pouquinho de lugar.
            # Antes usávamos AvgPool (média). O MaxPool escolhe sempre o pixel mais 'forte' (valor mais alto) da janela.
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # 2ª CAMADA CONVOLUCIONAL:
            # Agora recebemos os 6 "mapas" da camada anterior (in_channels=6) 
            # e aplicamos 16 filtros ainda mais complexos (out_channels=16).
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1),
            nn.ReLU(),
            
            # 2º POOLING:
            # Reduz as dimensões mais uma vez para condensar a essência da imagem pegando seus pontos principais.
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # --- 2. CLASSIFICAÇÃO (CLASSIFIER) ---
        # Aqui recebemos as características concentradas e purificadas pelas convoluções do passo 1
        # e usamos camadas densas (Lineares) para decidir qual classe (por ex, que dígito) está na imagem.
        self.classifier = nn.Sequential(
            
            # 1ª CAMADA LINEAR (Densa):
            # As convoluções e o pooling do passo anterior retornam pequenas "imagens" 2D (16 mapas de 5x5 pixels). 
            # Mas camadas Lineares precisam de um vetor plano e contínuo (em linha / modelo 1D). 
            # Então nós multiplicamos o tamanho total num achatamento: 16 * 5 * 5 = 400 valores numéricos.
            # A partir de 400 pontuações numéricas, a lógica matemática foca em apenas 120 na saída da camada.
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(),
            
            # 2ª CAMADA LINEAR:
            # Peneira e condensa as informações ainda mais, transformando 120 conceitos base em 84.
            nn.Linear(120, 84),
            nn.ReLU(),
            
            # 3ª CAMADA LINEAR (Saída Final):
            # Transforma os últimos 84 valores no número final que designamos para turmas diferentes (saídas).
            # Como a LeNet-5 foi criada para prever os dígitos (0 a 9) manuscritos, ela usa "num_classes=10".
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        """
        A função `forward` dita exatamente como a matriz de dados flui através da rede na hora de treinar e prever.
        Para toda imagem `x` que alimentamos o PyTorch, ele executa os passos matemáticos abaixo na ordem declarada:
        """
        
        # PASSO 1: Passa a imagem bruta pela estrutura de extração visual das Convoluções / Pooling
        x = self.features(x)
        
        # PASSO 2: Achatamento da Matriz 2D/3D (Flatten)
        # Transforma o formato 4D [Lote, 16, 5, 5] resultante da Etapa 1 num plano 2D continuo [Lote, 400]
        # O termo "1" diz ao PyTorch: "Mantenha o tamanho do Lote (Batch) como a dimensão principal para o agrupamento, e achate o resto".
        x = torch.flatten(x, 1)
        
        # PASSO 3: Passa a linha sequencialmente achatada pelo cérebro classificador (Camadas Lineares) e gera o resultado.
        logits = self.classifier(x)
        
        # Os "logits" são os números brutos originais previstos pelo nosso modelo.
        # Usualmente transformamos isso em probabilidades (de 0 a 1 em uma curva) depois usando matemática de Softmax ou Log-Loss.
        return logits


ModuleNotFoundError: No module named 'torch'

---
### 🖼️ Visualizando o Estado Inicial dos Filtros

Antes do treinamento ser executado, os pesos (filtros convolucionais) de uma rede em PyTorch recebem valores aleatórios de inicialização.

Abaixo nós instanciamos o modelo e usamos a biblioteca `matplotlib` para ver as matrizes dos 6 filtros presentes na nossa 1ª camada de convolução.

In [4]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Instanciar nossa rede (antes de ser treinada)
model = LeNet5()

# 2. Acessar os pesos correspondentes à primeira convolução (camada features[0])
# O bloco model.features[0] resgata o que definimos como nn.Conv2d(1, 6, kernel_size=5)
# Usamos .detach() porque não precisamos rastrear gradientes para visualização.
filters = model.features[0].weight.detach().cpu().numpy()

# As dimensões de um peso de Conv2d são: [out_channels, in_channels, altura, largura]
# Sendo assim, nosso shape será: (6, 1, 5, 5)
print(f"Formato do tensor dos filtros obtido: {filters.shape}")
print("Existem 6 filtros diferentes, cada um com tamanho 5x5 aplicados no canal de entrada.\n")

# 3. Criar uma renderização visual de cada filtro contido na camada
fig, axes = plt.subplots(1, 6, figsize=(15, 3))

for idx, ax in enumerate(axes):
    # Escolhemos o filtro de índice 'idx', vindo do primeiro canal de cor (0)
    f = filters[idx, 0, :, :]
    
    # Exibe matematicamente a intensidade dos pesos como uma imagem visual
    im = ax.imshow(f, cmap='viridis')
    ax.set_title(f'Filtro {idx+1}')
    ax.axis('off')

plt.suptitle('Estado dos Filtros (1ª Camada) Antes do Treino', fontsize=16)
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'